# DataSnake — Pipeline Defect Detection Model Training
**Sprint 2 · `datasnake-pipeline-v1`**

This notebook trains DataSnake's own YOLOv8-nano model on the Roboflow
"Corrosion Detection in Subsea Pipelines" dataset. The trained weights are
DataSnake IP — distributable to any client, improvable over time independently
of any third-party API.

**What you get at the end:**
- `datasnake-pipeline-v1.pt` — PyTorch weights (keep a backup)
- `datasnake-pipeline-v1.onnx` — ONNX export for Edge Function deployment
- `results.json` — per-image predictions (confidence, class, bbox) for all test images

**Runtime:** ~15 min on Colab free T4 GPU.

**After training:** run `ml/import_inference_results.py` to seed the real predictions
into `model_inference_log` in Supabase.


## Step 1 — Install dependencies

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## Step 2 — Download the dataset from Roboflow

Get your free API key at [roboflow.com](https://roboflow.com) → Settings → API.
The dataset files (images + YOLO annotations) are downloaded locally —
no Roboflow dependency after this step.

In [ ]:
ROBOFLOW_API_KEY = ""  # ← paste your key here

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("corrosion-detection-zvmog").project("corrosion-detection-in-subsea-pipelines")
version = project.version(1)
dataset = version.download("yolov8")

print(f"Dataset location: {dataset.location}")
DATASET_YAML = dataset.location + "/data.yaml"
print(f"data.yaml: {DATASET_YAML}")

## Step 3 — Train YOLOv8-nano (DataSnake-Pipeline-v1)

YOLOv8-nano is the smallest YOLOv8 variant — fast to train, easy to deploy
to an edge function. We fine-tune from ImageNet pre-trained weights.

Training takes ~15 min on a free Colab T4 GPU.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # pre-trained on COCO

results = model.train(
    data=DATASET_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    name="datasnake-pipeline-v1",
    project="datasnake-runs",
    patience=10,       # early stopping if no improvement for 10 epochs
    save=True,
    device=0,          # GPU
    verbose=True,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

## Step 4 — Evaluate on the test split

In [ ]:
import os
from pathlib import Path

best_weights = Path(results.save_dir) / "weights" / "best.pt"
model_best = YOLO(str(best_weights))

metrics = model_best.val(data=DATASET_YAML, split="test")
map50   = metrics.box.map50
map50_95 = metrics.box.map
print(f"\n{'='*50}")
print(f"DataSnake-Pipeline-v1 Test Metrics")
print(f"  mAP@0.5:      {map50:.4f}  ({map50*100:.1f}%)")
print(f"  mAP@0.5:0.95: {map50_95:.4f}")
print(f"{'='*50}")

## Step 5 — Run inference on all test images, generate results.json

This file is imported by `ml/import_inference_results.py` to seed real
predictions into `model_inference_log` in Supabase.

In [ ]:
import json
import glob
from pathlib import Path

dataset_dir = Path(dataset.location)
test_images = sorted(
    list(dataset_dir.glob("test/images/*.jpg")) +
    list(dataset_dir.glob("test/images/*.jpeg")) +
    list(dataset_dir.glob("test/images/*.png"))
)
print(f"Running inference on {len(test_images)} test images...")

inference_results = []

CLASS_NAMES = model_best.names  # dict {0: 'corrosion', ...}

for img_path in test_images:
    preds = model_best.predict(str(img_path), conf=0.25, verbose=False)
    pred  = preds[0]
    W, H  = pred.orig_shape[1], pred.orig_shape[0]

    if len(pred.boxes) == 0:
        # No detection — record as 'intact' with low confidence
        inference_results.append({
            "filename":       img_path.name,
            "detected_class": "intact",
            "confidence":     0.30,
            "bbox_pred":      None,
            "image_w":        W,
            "image_h":        H,
        })
        continue

    # Take highest-confidence detection
    best_box = max(pred.boxes, key=lambda b: float(b.conf[0]))
    cls_idx  = int(best_box.cls[0])
    conf     = float(best_box.conf[0])
    # xyxy format → normalised
    x1, y1, x2, y2 = best_box.xyxy[0].tolist()
    bbox_norm = [
        round(x1 / W, 4), round(y1 / H, 4),
        round(x2 / W, 4), round(y2 / H, 4),
    ]

    inference_results.append({
        "filename":       img_path.name,
        "detected_class": CLASS_NAMES.get(cls_idx, f"class_{cls_idx}"),
        "confidence":     round(conf, 4),
        "bbox_pred":      bbox_norm,
        "image_w":        W,
        "image_h":        H,
    })

out_path = Path("results.json")
out_path.write_text(json.dumps(inference_results, indent=2))
print(f"\nSaved {len(inference_results)} predictions to {out_path}")

# Quick summary
from collections import Counter
cls_counts = Counter(r["detected_class"] for r in inference_results)
print("\nClass distribution in test predictions:")
for cls, count in cls_counts.most_common():
    print(f"  {cls:20s} {count:3d} images  avg conf: "
          f"{sum(r['confidence'] for r in inference_results if r['detected_class']==cls)/count:.3f}")

## Step 6 — Export to ONNX

The ONNX model is DataSnake's deployable artifact. Upload it to Supabase
Storage bucket `ml-models` as `datasnake-pipeline-v1.onnx`.
Sprint 3 will load it from a Supabase Edge Function for live inference.

In [ ]:
onnx_path = model_best.export(format="onnx", dynamic=True, simplify=True)
print(f"\nONNX model exported to: {onnx_path}")

# Rename to the DataSnake versioned name
import shutil
final_name = "datasnake-pipeline-v1.onnx"
shutil.copy(onnx_path, final_name)
print(f"Copied to: {final_name}")
print(f"\nNext steps:")
print(f"  1. Download results.json and {final_name} from this session")
print(f"  2. Run: python ml/import_inference_results.py --results results.json")
print(f"  3. Upload {final_name} to Supabase Storage bucket 'ml-models'")

## Step 7 — Download outputs from Colab

Download these files to your local machine, then push them to Supabase.

In [ ]:
from google.colab import files

print("Downloading results.json...")
files.download("results.json")

print("Downloading datasnake-pipeline-v1.onnx...")
files.download("datasnake-pipeline-v1.onnx")

# Also download the PyTorch weights as a backup
import shutil
shutil.copy(str(best_weights), "datasnake-pipeline-v1.pt")
print("Downloading datasnake-pipeline-v1.pt (PyTorch weights backup)...")
files.download("datasnake-pipeline-v1.pt")

---
## Done — DataSnake-Pipeline-v1 trained ✓

After downloading the outputs:

```bash
# 1. Import real predictions into Supabase
python ml/import_inference_results.py --results results.json

# 2. Upload ONNX weights to Supabase Storage
#    (via Supabase dashboard → Storage → ml-models → Upload)
#    or via the CLI:
supabase storage cp datasnake-pipeline-v1.onnx ss:///ml-models/
```

The model is now DataSnake's IP. It can be:
- Retrained on more data (add more images → re-run this notebook)
- Distributed to any client (they get the `.onnx` file + Edge Function)
- Versioned (`datasnake-pipeline-v2`, `v3`, ...) as quality improves
